# Colab vs 2HDMC clean discrepancy analysis

This is not a lambda drift analysis. Focus is channel/configuration discrepancy.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')
root = Path.cwd()
candidate = root/'scripts/out/refactor_colab_compare/real_run'
if not candidate.exists():
    candidate = root.parent/'scripts/out/refactor_colab_compare/real_run'
BASE = candidate
FIG_DIR = BASE.parent/'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
merged=pd.read_csv(BASE/'colab_vs_2hdmc_merged_comparison.csv')
width_stats=pd.read_csv(BASE/'width_rel_stats.csv')
grouped=pd.read_csv(BASE/'grouped_error_by_l1_l6_tb.csv')
print(len(merged), int(merged['set_param_ok'].sum()), int(merged['triple_ok'].sum()), int(merged['warning_flag'].sum()))
print('warning_flag caveat: not automatic physics veto')
df=merged[merged['triple_ok']==True].copy() if 'triple_ok' in merged.columns else merged.copy()


In [ ]:
ordered=width_stats.sort_values('median_rel',ascending=False)
plt.figure(figsize=(8,4)); sns.barplot(data=ordered,x='channel',y='median_rel',color='#4C78A8'); plt.tight_layout(); plt.savefig(FIG_DIR/'channel_median_rel_errors.png',dpi=160)
plt.figure(figsize=(8,4)); sns.barplot(data=ordered,x='channel',y='max_rel',color='#F58518'); plt.tight_layout(); plt.savefig(FIG_DIR/'channel_max_rel_errors.png',dpi=160)
ordered[['channel','mean_rel','median_rel','max_rel']]

In [ ]:
show=grouped.sort_values('median_rel_total',ascending=False)
plt.figure(figsize=(7,4)); sns.barplot(data=show,x='lambda6',y='median_rel_total',hue='lambda1'); plt.tight_layout(); plt.savefig(FIG_DIR/'grouped_config_median_rel_total.png',dpi=160)
fig,ax=plt.subplots(1,2,figsize=(12,4)); sns.barplot(data=show,x='lambda6',y='median_rel_gg',hue='lambda1',ax=ax[0]); sns.barplot(data=show,x='lambda6',y='median_rel_gaga',hue='lambda1',ax=ax[1]); plt.tight_layout(); plt.savefig(FIG_DIR/'grouped_config_median_rel_gg_gaga.png',dpi=160)
show[['lambda1','lambda6','tan_beta','n','median_rel_gg','median_rel_gaga','median_rel_total']]

In [ ]:
long_cols=[c for c in ['rel_err_width_bb','rel_err_width_cc','rel_err_width_tautau','rel_err_width_gg','rel_err_width_gaga','rel_err_width_Zga','rel_err_width_total'] if c in df.columns]
long=df[long_cols].melt(var_name='channel',value_name='rel_err'); long['channel']=long['channel'].str.replace('rel_err_width_','',regex=False)
plt.figure(figsize=(10,4)); sns.boxplot(data=long,x='channel',y='rel_err'); plt.ylim(0,np.nanpercentile(long['rel_err'],99.5)); plt.tight_layout()
print('tautau median', float(df['rel_err_width_tautau'].median()))

In [ ]:
plt.figure(figsize=(8,4)); sns.scatterplot(data=df,x='mH',y='rel_err_width_gg',hue='lambda6',style='lambda1',alpha=0.5,s=20); plt.tight_layout(); plt.savefig(FIG_DIR/'mH_vs_rel_err_gg.png',dpi=160)
plt.figure(figsize=(8,4)); sns.scatterplot(data=df,x='mH',y='rel_err_width_gaga',hue='lambda6',style='lambda1',alpha=0.5,s=20); plt.tight_layout(); plt.savefig(FIG_DIR/'mH_vs_rel_err_gaga.png',dpi=160)

In [ ]:
from pandas.plotting import parallel_coordinates
cols=['mH','lambda1','lambda6','tan_beta','rel_err_width_gg','rel_err_width_gaga','rel_err_width_total']
sub=df[cols].sample(min(200,len(df)),random_state=42).copy(); sub['color_bin']=pd.qcut(sub['rel_err_width_total'],q=4,labels=['q1','q2','q3','q4'])
for c in cols:
 v=sub[c].astype(float); d=v.max()-v.min(); sub[c]=0.0 if d==0 else (v-v.min())/d
plt.figure(figsize=(12,5)); parallel_coordinates(sub[['color_bin']+cols],'color_bin',colormap='viridis',alpha=0.2); plt.tight_layout(); plt.savefig(FIG_DIR/'parallel_hyperparams_errors.png',dpi=160)

In [ ]:
top_cols=[c for c in ['point_id','mH','lambda1','lambda6','tan_beta','rel_err_width_gg','rel_err_width_gaga','rel_err_width_Zga','rel_err_width_total','width_gg','width_gg_colab','total_width','total_width_colab'] if c in df.columns]
df.sort_values('rel_err_width_total',ascending=False)[top_cols].head(20)

## Final interpretation
- gg discrepancy is systematic and dominant.
- tautau is a stable control channel.
- gaga/total are lambda6-sensitive.
- no lambda1 drift claim.
- warning_flag needs separate audit before veto usage.